# Notebook 03 — Model Training

Trains XGBoost, evaluates performance, saves artifacts to `ml/model_artifacts/`.

**Run this notebook in Colab, then push artifacts to GitHub.**

In [ ]:
# ── Install & imports ─────────────────────────────────────────────────────
!pip install xgboost scikit-learn pandas numpy shap matplotlib -q

import pandas as pd
import numpy as np
import pickle, os
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score, roc_auc_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc
)
from xgboost import XGBClassifier
import shap

plt.style.use('dark_background')
os.makedirs('ml/model_artifacts', exist_ok=True)
print('Setup complete ✓')

In [ ]:
# ── Clone repo (run once in Colab) ────────────────────────────────────────
# !git clone https://github.com/YOUR_USERNAME/customer-churn-predictor.git
# %cd customer-churn-predictor

# ── Load + clean ──────────────────────────────────────────────────────────
df = pd.read_csv('data/raw/telco_churn.csv')
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)
df.drop(columns=['customerID'], inplace=True)
df['Churn'] = (df['Churn'] == 'Yes').astype(int)
print(f'Loaded: {df.shape}  |  Churn rate: {df["Churn"].mean():.1%}')

In [ ]:
# ── Feature engineering ───────────────────────────────────────────────────
df['charges_per_tenure'] = df['MonthlyCharges'] / (df['tenure'] + 1)

service_cols = [
    'PhoneService','MultipleLines','InternetService',
    'OnlineSecurity','OnlineBackup','DeviceProtection',
    'TechSupport','StreamingTV','StreamingMovies'
]
df['service_count'] = df[service_cols].apply(
    lambda row: sum(1 for v in row if v not in ['No','No internet service','No phone service']),
    axis=1
)
df['is_long_term'] = (df['tenure'] > 24).astype(int)

print('New features added: charges_per_tenure, service_count, is_long_term')

In [ ]:
# ── Encode categoricals ───────────────────────────────────────────────────
le_dict = {}
for col in df.select_dtypes(include='object').columns:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    le_dict[col] = le
print(f'Encoded {len(le_dict)} categorical columns')

In [ ]:
# ── Scale numerics ────────────────────────────────────────────────────────
num_cols = ['tenure','MonthlyCharges','TotalCharges','charges_per_tenure','service_count']
scaler = StandardScaler()
df[num_cols] = scaler.fit_transform(df[num_cols])

X = df.drop(columns=['Churn'])
y = df['Churn']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')

In [ ]:
# ── Train XGBoost ─────────────────────────────────────────────────────────
model = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=(y_train==0).sum()/(y_train==1).sum(),
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1
)
model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=50)
print('Training complete ✓')

In [ ]:
# ── Evaluation ────────────────────────────────────────────────────────────
y_pred  = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:,1]

acc = accuracy_score(y_test, y_pred)
auc_score = roc_auc_score(y_test, y_proba)

print(f'Accuracy : {acc:.4f}')
print(f'ROC-AUC  : {auc_score:.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=['Stay','Churn']))

In [ ]:
# ── Cross-validation ──────────────────────────────────────────────────────
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(model, X, y, cv=cv, scoring='roc_auc', n_jobs=-1)
print(f'5-Fold CV ROC-AUC: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

In [ ]:
# ── Confusion matrix + ROC curve ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=['Stay','Churn'])
disp.plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Confusion Matrix')

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_proba)
axes[1].plot(fpr, tpr, color='#6c63ff', lw=2, label=f'ROC AUC = {auc_score:.3f}')
axes[1].plot([0,1],[0,1], 'k--', lw=1, alpha=0.5)
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve')
axes[1].legend()

plt.tight_layout(); plt.show()

In [ ]:
# ── Feature importance ────────────────────────────────────────────────────
importances = pd.Series(model.feature_importances_, index=X.columns)
importances.sort_values().tail(15).plot(
    kind='barh', color='#a78bfa', edgecolor='none', figsize=(9,6)
)
plt.title('XGBoost Feature Importance (top 15)')
plt.xlabel('Importance Score')
plt.tight_layout(); plt.show()

In [ ]:
# ── Save all artifacts ────────────────────────────────────────────────────
pickle.dump(model,             open('ml/model_artifacts/xgb_model.pkl',       'wb'))
pickle.dump(scaler,            open('ml/model_artifacts/scaler.pkl',           'wb'))
pickle.dump(le_dict,           open('ml/model_artifacts/label_encoders.pkl',   'wb'))
pickle.dump(list(X.columns),   open('ml/model_artifacts/feature_names.pkl',    'wb'))

# Save processed data
os.makedirs('data/processed', exist_ok=True)
df.to_csv('data/processed/churn_processed.csv', index=False)

print('Artifacts saved:')
for f in os.listdir('ml/model_artifacts'):
    size = os.path.getsize(f'ml/model_artifacts/{f}') / 1024
    print(f'  ✓ {f:35s} {size:.1f} KB')

In [ ]:
# ── Push to GitHub (run in Colab) ─────────────────────────────────────────
# 1. Get your token: github.com → Settings → Developer Settings → Personal Access Tokens
# 2. Store it in Colab Secrets (left sidebar → key icon) as GITHUB_TOKEN

# from google.colab import userdata
# GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
# REPO = 'YOUR_USERNAME/customer-churn-predictor'
#
# !git config user.email 'your@email.com'
# !git config user.name  'Your Name'
# !git remote set-url origin https://{GITHUB_TOKEN}@github.com/{REPO}.git
# !git add ml/model_artifacts/ data/processed/
# !git commit -m 'add trained model artifacts'
# !git push origin main
print('Uncomment the block above and fill in YOUR details to push to GitHub')